# Observability with OpenTelemetry

Pixeltable can export its internal telemetry as OpenTelemetry (OTel) spans and metrics, so you can watch
operations in any OTel backend. This notebook wires the OTel bridge to **Grafana Cloud** and traces an
`insert()`.

An insert emits a nested span tree:

```
pixeltable.insert            (operation span, root)
└─ pixeltable.row            (one per inserted row, DEBUG level)
   └─ udf.<name>             (each UDF call, nested under its row)
```

Traces land in Grafana Cloud Traces (Tempo); the notebook also shows the metrics path
(`pixeltable.rows.written`) reaching Grafana.

In [ ]:
%pip install -qU 'pixeltable[otel]'

## Turn on the OpenTelemetry bridge

Telemetry is opt-in: you must call `pxt_otel.init()` once. It reads the standard `OTEL_EXPORTER_OTLP_*`
env vars if present, otherwise we build the endpoint and Basic-auth header from the `GRAFANA_*` vars.

`init()` runs **once per process**. If it ran under a bad config earlier in this kernel it can't be
reconfigured, so restart the kernel and run this cell first.

In [ ]:
import base64
import os

import opentelemetry.instrumentation.pixeltable as pxt_otel
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from pixeltable import exceptions as excs, hooks

# init() is ONCE PER PROCESS. If it already ran in this kernel under the wrong config, re-running is a
# no-op (it raises below) -- restart the kernel (Kernel -> Restart) to reconfigure.
try:
    if os.environ.get('OTEL_EXPORTER_OTLP_ENDPOINT'):
        pxt_otel.init()  # resolve endpoint/headers natively from the OTEL_EXPORTER_OTLP_* env vars
    else:
        # configure explicitly from the GRAFANA_* vars -- does not rely on the kernel inheriting OTEL_* vars
        auth = base64.b64encode(
            f'{os.environ["GRAFANA_OTLP_INSTANCE_ID"]}:{os.environ["GRAFANA_SERVICE_ACCOUNT_TOKEN"]}'.encode()
        ).decode()
        pxt_otel.init(endpoint=os.environ['GRAFANA_OTLP_URL'], headers=f'Authorization=Basic {auth}')
except excs.Error as e:
    print(f'init() skipped: {e}')

hooks.set_span_level(hooks.DEBUG)  # emit per-row + per-UDF spans (default INFO = operation span only)

# fail loud instead of silently dropping every span: a no-op ProxyTracerProvider means the SDK never came up
assert isinstance(trace.get_tracer_provider(), TracerProvider), (
    'OpenTelemetry is INERT -- no OTLP exporter was configured on the first init() in this kernel. '
    'Restart the kernel and run this cell FIRST, before any table operation.'
)
print('OTel bridge active -> Grafana Cloud')

## A table with a UDF-backed computed column

The computed column runs `add_one` on every inserted row, producing the `udf.add_one` spans.

In [ ]:
import pixeltable as pxt


@pxt.udf
def add_one(x: int) -> int:
    return x + 1


t = pxt.create_table('otel_demo', {'c': pxt.Int}, if_exists='replace')
t.add_computed_column(inc=add_one(t.c))

## Insert

This is the traced operation: 10 rows (under the 100-span-per-operation cap, so every row gets a span).

In [ ]:
status = t.insert([{'c': i} for i in range(10)])
status

## Wait a few seconds for export

`init()` sets up a `BatchSpanProcessor`, which ships queued spans automatically about every 5 seconds while the kernel stays alive, so no manual flush is needed. Give it a few seconds after the insert, then check Grafana.

(In a short-lived script that exits immediately you would call `trace.get_tracer_provider().force_flush()` before exit, but in a live notebook kernel the timer handles it.)

## View it in Grafana

- **Traces**: Explore -> your Traces (Tempo) data source -> search Service Name `pixeltable`, or span name
  `pixeltable.insert`. Expand a trace to see `pixeltable.row` -> `udf.add_one` nesting.
- **Metrics**: Explore -> your Prometheus data source -> query `pixeltable_rows_written_total`.

For more detail raise the span level (`hooks.set_span_level(hooks.TRACE)`); to reduce volume on a large
insert, leave it at the default INFO (operation + per-batch spans only).